# Moli OS — votre clé bêta, une seule fois
Ouvrez ce carnet dans **votre compte Google, Colab gratuit, CPU**. Aucun Drive à monter, abonnement ou GPU nécessaire. Le code tourne dans une VM privée de votre compte Google, pas chez Codex. Ne partagez pas cette session.

1. Dans NordPass Android, créez **Moli OS — Signature bêta** avec un mot de passe aléatoire de 32 caractères ASCII. S’il contient déjà un fichier .p12, **ne recréez pas de clé** : exécutez **0 — Initialiser**, puis **2 — Vérifier depuis NordPass**, sans lancer la création.
2. Exécutez **0 — Initialiser** (aucune clé créée), puis **1 — Créer une fois** : créez/téléchargez le `.p12`, joignez-le à cette entrée NordPass et notez son empreinte publique.
3. **Retéléchargez la pièce jointe depuis NordPass**, puis vérifiez ce fichier dans **2 — Vérifier depuis NordPass**. Saisissez l’empreinte publique conservée dans NordPass, même dans une nouvelle session. Le bouton de copie sert uniquement au secret GitHub `BETA_SIGNING_BUNDLE`, jamais au chat/issue/PR. Le carnet ne configure pas GitHub et ne signe aucune APK.

Ne choisissez jamais « Enregistrer dans GitHub ». Les sorties privées sont désactivées dans les sauvegardes du carnet. Fermez/supprimez la VM après le transfert ; les fichiers de téléchargement personnels restent à gérer sur votre appareil après vérification de NordPass.


In [ ]:
#@title 0 — Initialiser (aucune création de clé)
"""Owner-only PKCS12 helper. Never run to generate a user's key in Codex/CI."""
import base64
from datetime import datetime, timedelta, timezone
import json
import re
from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives.serialization import pkcs12
from cryptography.x509.oid import NameOID


def password_bytes(password):
    if not isinstance(password, str) or not 24 <= len(password) <= 128 or any(not 33 <= ord(c) <= 126 for c in password):
        raise ValueError('Utilisez un mot de passe NordPass aléatoire de 32 caractères ASCII.')
    return password.encode('ascii')


def create_key(password):
    secret = password_bytes(password)
    key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
    name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Moli OS Beta')])
    now = datetime.now(timezone.utc)
    certificate = (x509.CertificateBuilder().subject_name(name).issuer_name(name)
                   .public_key(key.public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=1)).not_valid_after(now + timedelta(days=10000))
                   .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
                   .sign(key, hashes.SHA256()))
    blob = pkcs12.serialize_key_and_certificates(b'moli-beta', key, certificate, None, serialization.BestAvailableEncryption(secret))
    return blob, certificate.fingerprint(hashes.SHA256()).hex()


def recovery_bundle(blob, password, expected_fingerprint, *, nordpass_confirmed):
    if nordpass_confirmed is not True:
        raise ValueError('Retéléchargez et vérifiez le fichier depuis votre entrée NordPass.')
    if not re.fullmatch('[a-f0-9]{64}', expected_fingerprint) or not 1000 <= len(blob) <= 20000:
        raise ValueError('Fichier ou empreinte invalide.')
    try:
        key, certificate, chain = pkcs12.load_key_and_certificates(blob, password_bytes(password))
        if not isinstance(key, rsa.RSAPrivateKey) or key.key_size < 3072 or certificate is None or chain:
            raise ValueError()
        if key.public_key().public_numbers() != certificate.public_key().public_numbers():
            raise ValueError()
        if certificate.fingerprint(hashes.SHA256()).hex() != expected_fingerprint:
            raise ValueError()
    except Exception:
        raise ValueError('Le fichier, le mot de passe ou l’empreinte ne correspond pas.') from None
    return json.dumps({'format': 1, 'p12': base64.b64encode(blob).decode('ascii'), 'password': password,
                       'certificate_sha256': expected_fingerprint, 'nordpass_confirmed': True}, separators=(',', ':'))


In [ ]:
#@title 1 — Créer et télécharger la toute première clé (une seule fois)
# Si NordPass possède déjà le .p12, NE PAS relancer une création : utilisez la cellule 2.
import getpass, os
from pathlib import Path
from google.colab import files

if '_moli_pending_p12' not in globals():
    target = Path('/content/moli-beta-signature.p12')
    if target.exists():
        raise ValueError('Un fichier existe déjà : ne pas remplacer une clé. Utilisez la vérification.')
    password = getpass.getpass('Mot de passe NordPass (masqué) : ')
    confirmation = getpass.getpass('Le même mot de passe : ')
    if password != confirmation:
        raise ValueError('Les mots de passe diffèrent.')
    _moli_pending_p12, _moli_fingerprint = create_key(password)
    del password, confirmation
    fd = os.open(target, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, 'wb') as stream:
        stream.write(_moli_pending_p12)
print('Empreinte publique à noter dans NordPass :', _moli_fingerprint)
files.download('/content/moli-beta-signature.p12')


In [ ]:
#@title 2 — Vérifier le fichier récupéré de NordPass et préparer la copie GitHub
# Ne lancez cette cellule qu’après avoir joint puis retéléchargé le .p12 depuis NordPass.
import getpass, json
from google.colab import files, output

if input('Fichier retéléchargé depuis NordPass ? Tapez OUI : ').strip() != 'OUI':
    raise ValueError('Sauvegarde personnelle requise ; aucune signature activée.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Sélectionnez uniquement le .p12 récupéré depuis NordPass.')
blob = next(iter(uploaded.values()))
expected = input('Empreinte publique notée dans NordPass : ').strip().lower().replace(':','')
password = getpass.getpass('Mot de passe NordPass (masqué) : ')
bundle = recovery_bundle(blob, password, expected, nordpass_confirmed=True)
print('Copie récupérée et mot de passe vérifiés. Empreinte publique :', expected)
# DOM éphémère, hors sorties sérialisées : aucune impression de clé/base64/mot de passe.
js = """new Promise((resolve) => {
 const box = document.createElement('div');
 box.style.cssText='position:fixed;inset:12%;padding:24px;background:white;color:black;z-index:2147483647;border:2px solid #15616a;overflow:auto';
 const text = document.createElement('p'); text.textContent='Copiez uniquement dans GitHub → Environment android-beta-signing → Secret BETA_SIGNING_BUNDLE. Jamais dans le chat, une issue ou un fichier partagé.';
 const button = document.createElement('button'); button.textContent='Copier pour le secret GitHub';
 const cancel = document.createElement('button'); cancel.textContent='Fermer sans copier';
 box.append(text,button,cancel); document.body.append(box);
 button.onclick=async()=>{try{await navigator.clipboard.writeText(BUNDLE);box.remove();resolve(true);}catch(e){text.textContent='Copie non autorisée par le navigateur. Réessayez dans Chrome ; aucun secret ne sera affiché.';}};
 cancel.onclick=()=>{box.remove();resolve(false);};
})"""
copied = output.eval_js(js.replace('BUNDLE', json.dumps(bundle)))
del bundle, password, blob, uploaded
print('Copie effectuée : collez uniquement dans le secret GitHub, puis remplacez le presse-papiers.' if copied else 'Aucune copie. Le .p12 conservé dans NordPass permet de reprendre plus tard sans recréer de clé.')
